In [1]:
import xarray as xr

# Specify the file path and the engine
ds = xr.open_dataset("2015.grib", engine="cfgrib")

# To handle potential issues with complex or heterogeneous files,
# you might need to use open_datasets which returns a list of datasets
# or specify filtering options.
# ds_list = cfgrib.open_datasets("path/to/yourfile.grib2")

In [2]:
ds

<xarray.Dataset> Size: 3MB
Dimensions:        (time: 1460, isobaricInhPa: 6, latitude: 7, longitude: 7)
Coordinates:
  * time           (time) datetime64[ns] 12kB 2015-01-01 ... 2015-12-31T18:00:00
    valid_time     (time) datetime64[ns] 12kB ...
  * isobaricInhPa  (isobaricInhPa) float64 48B 1e+03 925.0 850.0 ... 500.0 300.0
  * latitude       (latitude) float64 56B 48.5 48.25 48.0 47.75 47.5 47.25 47.0
  * longitude      (longitude) float64 56B -53.5 -53.25 -53.0 ... -52.25 -52.0
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
Data variables:
    q              (time, isobaricInhPa, latitude, longitude) float32 2MB ...
    t              (time, isobaricInhPa, latitude, longitude) float32 2MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-16T23:39 GRIB to CDM+CF via cfgrib-0.9.1...

```
======================================================================
PRESSURE DATA STRUCTURE
======================================================================
<xarray.Dataset> Size: 18GB
Dimensions:        (time: 13148, isobaricInhPa: 6, latitude: 81, longitude: 361)
Coordinates:
  * time           (time) datetime64[ns] 105kB 2015-01-01 ... 2023-12-31T18:0...
    valid_time     (time) datetime64[ns] 105kB dask.array<chunksize=(1460,), meta=np.ndarray>
  * isobaricInhPa  (isobaricInhPa) float64 48B 1e+03 975.0 850.0 ... 500.0 300.0
  * latitude       (latitude) float64 648B 60.0 59.75 59.5 ... 40.5 40.25 40.0
  * longitude      (longitude) float64 3kB -140.0 -139.8 -139.5 ... -50.25 -50.0
    number         int64 8B 0
    step           timedelta64[ns] 8B 00:00:00
Data variables:
    q              (time, isobaricInhPa, latitude, longitude) float32 9GB dask.array<chunksize=(1460, 6, 81, 361), meta=np.ndarray>
    t              (time, isobaricInhPa, latitude, longitude) float32 9GB dask.array<chunksize=(1460, 6, 81, 361), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-16T23:33 GRIB to CDM+CF via cfgrib-0.9.1...

Variables: ['q', 't']
Coordinates: ['number', 'time', 'step', 'isobaricInhPa', 'latitude', 'longitude', 'valid_time']
Shape: (13148, 6, 81, 361)

======================================================================
SURFACE DATA STRUCTURE
======================================================================
<xarray.Dataset> Size: 5GB
Dimensions:     (time: 13148, latitude: 81, longitude: 361)
Coordinates:
  * time        (time) datetime64[ns] 105kB 2015-01-01 ... 2023-12-31T18:00:00
    valid_time  (time) datetime64[ns] 105kB dask.array<chunksize=(1460,), meta=np.ndarray>
  * latitude    (latitude) float64 648B 60.0 59.75 59.5 ... 40.5 40.25 40.0
  * longitude   (longitude) float64 3kB -140.0 -139.8 -139.5 ... -50.25 -50.0
    number      int64 8B 0
    step        timedelta64[ns] 8B 00:00:00
    surface     float64 8B 0.0
Data variables:
    t2m         (time, latitude, longitude) float32 2GB dask.array<chunksize=(1460, 81, 361), meta=np.ndarray>
    d2m         (time, latitude, longitude) float32 2GB dask.array<chunksize=(1460, 81, 361), meta=np.ndarray>
    sp          (time, latitude, longitude) float32 2GB dask.array<chunksize=(1460, 81, 361), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-16T23:36 GRIB to CDM+CF via cfgrib-0.9.1...

Variables: ['t2m', 'd2m', 'sp']
Coordinates: ['number', 'time', 'step', 'surface', 'latitude', 'longitude', 'valid_time']
Shape: (13148, 81, 361)
```

In [3]:
# Load combined data
pressure_data = xr.open_dataset('era5_pressure_2015_2023.nc')
surface_data = xr.open_dataset('era5_surface_2015_2023.nc')

# Define locations
locations = {
    'St_Johns': (47.6, -52.7),
    'Toronto': (43.7, -79.4),
    'Vancouver': (49.3, -123.1),
}

print("Extracting 1D columns...\n")

for loc_name, (lat, lon) in locations.items():
    print(f"Extracting {loc_name} (lat={lat}, lon={lon})...")

    # Select nearest grid point
    p_col = pressure_data.sel(latitude=lat, longitude=lon, method='nearest')
    s_col = surface_data.sel(latitude=lat, longitude=lon, method='nearest')

    # Merge into single dataset
    column = xr.merge([p_col, s_col])

    # Compute dask arrays to memory (required before saving)
    column = column.compute()

    # Save as NetCDF
    column.to_netcdf(f'era5_column_{loc_name}.nc')

    print(f"✓ Saved era5_column_{loc_name}.nc")
    print(f"  Time steps: {len(column.time)}")
    print(f"  Variables: {list(column.data_vars)}")
    print(f"  Pressure levels: {list(column.isobaricInhPa.values)}")
    print(
        f"  Location: lat={column.latitude.values:.2f}, lon={column.longitude.values:.2f}\n")

print("✓ All columns extracted and saved!")

Extracting 1D columns...

Extracting St_Johns (lat=47.6, lon=-52.7)...


/var/folders/tl/52w5zxkd1zl65mr95qllj_3h0000gn/T/ipykernel_48033/2338375603.py:22: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  column = xr.merge([p_col, s_col])
/var/folders/tl/52w5zxkd1zl65mr95qllj_3h0000gn/T/ipykernel_48033/2338375603.py:22: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  column = xr.merge([p_col, s_col])
/var/folde

✓ Saved era5_column_St_Johns.nc
  Time steps: 13148
  Variables: ['q', 't', 't2m', 'd2m', 'sp']
  Pressure levels: [np.float64(1000.0), np.float64(975.0), np.float64(850.0), np.float64(700.0), np.float64(500.0), np.float64(300.0)]
  Location: lat=47.50, lon=-52.75

Extracting Toronto (lat=43.7, lon=-79.4)...


/var/folders/tl/52w5zxkd1zl65mr95qllj_3h0000gn/T/ipykernel_48033/2338375603.py:22: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  column = xr.merge([p_col, s_col])
/var/folders/tl/52w5zxkd1zl65mr95qllj_3h0000gn/T/ipykernel_48033/2338375603.py:22: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  column = xr.merge([p_col, s_col])
/var/folde

✓ Saved era5_column_Toronto.nc
  Time steps: 13148
  Variables: ['q', 't', 't2m', 'd2m', 'sp']
  Pressure levels: [np.float64(1000.0), np.float64(975.0), np.float64(850.0), np.float64(700.0), np.float64(500.0), np.float64(300.0)]
  Location: lat=43.75, lon=-79.50

Extracting Vancouver (lat=49.3, lon=-123.1)...


/var/folders/tl/52w5zxkd1zl65mr95qllj_3h0000gn/T/ipykernel_48033/2338375603.py:22: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  column = xr.merge([p_col, s_col])
/var/folders/tl/52w5zxkd1zl65mr95qllj_3h0000gn/T/ipykernel_48033/2338375603.py:22: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  column = xr.merge([p_col, s_col])
/var/folde

✓ Saved era5_column_Vancouver.nc
  Time steps: 13148
  Variables: ['q', 't', 't2m', 'd2m', 'sp']
  Pressure levels: [np.float64(1000.0), np.float64(975.0), np.float64(850.0), np.float64(700.0), np.float64(500.0), np.float64(300.0)]
  Location: lat=49.25, lon=-123.00

✓ All columns extracted and saved!


In [4]:
import xarray as xr

col = xr.open_dataset('era5_column_St_Johns.nc')
print(col)
print(f"\nTime range: {col.time.values[0]} to {col.time.values[-1]}")
print(f"Time steps: {len(col.time)}")
print(f"Variables: {list(col.data_vars)}")

<xarray.Dataset> Size: 999kB
Dimensions:        (time: 13148, isobaricInhPa: 6)
Coordinates:
  * time           (time) datetime64[ns] 105kB 2015-01-01 ... 2023-12-31T18:0...
    valid_time     (time) datetime64[ns] 105kB ...
  * isobaricInhPa  (isobaricInhPa) float64 48B 1e+03 975.0 850.0 ... 500.0 300.0
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
    latitude       float64 8B ...
    longitude      float64 8B ...
    surface        float64 8B ...
Data variables:
    q              (time, isobaricInhPa) float32 316kB ...
    t              (time, isobaricInhPa) float32 316kB ...
    t2m            (time) float32 53kB ...
    d2m            (time) float32 53kB ...
    sp             (time) float32 53kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             Eu

In [12]:
col['q'].coords

Coordinates:
  * time           (time) datetime64[ns] 105kB 2015-01-01 ... 2023-12-31T18:0...
    valid_time     (time) datetime64[ns] 105kB ...
  * isobaricInhPa  (isobaricInhPa) float64 48B 1e+03 975.0 850.0 ... 500.0 300.0
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
    latitude       float64 8B ...
    longitude      float64 8B ...
    surface        float64 8B ...